# Algorithme de Grover : Recherche non-structurée

$$|\psi\rangle = H^{\otimes n}|0\rangle \xrightarrow{G^k} |\omega\rangle$$

Opérateur de Grover: $G = (2|\psi\rangle\langle\psi| - I) \, O$ où $O$ est l'oracle.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, Aer, execute
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

### Oracle

$$O|x\rangle = \begin{cases} -|x\rangle & \text{si } x = \omega \\ |x\rangle & \text{sinon} \end{cases}$$

Marque l'état cible $\omega$ en inversant sa phase.

In [ ]:
def oracle(n, target):
    qc = QuantumCircuit(n, name='Oracle')
    for i in range(n):
        if not (target >> i) & 1:
            qc.x(i)
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    for i in range(n):
        if not (target >> i) & 1:
            qc.x(i)
    return qc.to_gate()

### Diffuseur

$$D = 2|\psi\rangle\langle\psi| - I = H^{\otimes n} (2|0\rangle\langle0| - I) H^{\otimes n}$$

Amplifie les états marqués par interférence constructive.

In [ ]:
def diffuser(n):
    qc = QuantumCircuit(n, name='Diffuseur')
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc.to_gate()

In [ ]:
def grover_circuit(n, target, n_iters):
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    for _ in range(n_iters):
        qc.append(oracle(n, target), range(n))
        qc.append(diffuser(n), range(n))
    qc.measure(range(n), range(n))
    return qc

In [ ]:
n = 4
target = 6
n_iters = int(np.pi / 4 * np.sqrt(2**n))
print(f'n={n}, cible={target}, itérations optimales={n_iters}')

qc_grover = grover_circuit(n, target, n_iters)
backend = Aer.get_backend('qasm_simulator')
counts = execute(qc_grover, backend, shots=2048).result().get_counts()
plot_histogram(counts, title=f'Grover n={n}, cible={target}')

In [ ]:
# Probabilité de l'état cible
p_target = counts.get(f'{target:0{n}b}', 0) / 2048
print(f'P(cible) = {p_target:.3f}')
print(f'Rapport: {max(counts.values()) / sum(counts.values()) * 100:.1f}%')

### Généralisation à $M$ solutions

Nombre d'itérations optimal:
$$k_{\text{opt}} = \left\lfloor \frac{\pi}{4}\sqrt{\frac{N}{M}} \right\rfloor$$

Probabilité maximale: $P_{\text{succès}} \approx 1 - \frac{M}{N}$.

In [ ]:
def oracle_multi(n, targets):
    qc = QuantumCircuit(n, name='Oracle')
    for t in targets:
        for i in range(n):
            if not (t >> i) & 1:
                qc.x(i)
        qc.h(n - 1)
        qc.mcx(list(range(n - 1)), n - 1)
        qc.h(n - 1)
        for i in range(n):
            if not (t >> i) & 1:
                qc.x(i)
    return qc.to_gate()

def grover_multi(n, targets):
    M = len(targets)
    k_opt = int(np.pi / 4 * np.sqrt(2**n / M))
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    for _ in range(k_opt):
        qc.append(oracle_multi(n, targets), range(n))
        qc.append(diffuser(n), range(n))
    qc.measure(range(n), range(n))
    return qc

n = 4
targets = [3, 7, 11]
qc_multi = grover_multi(n, targets)
counts_m = execute(qc_multi, backend, shots=2048).result().get_counts()
p_success = sum(counts_m.get(f'{t:0{n}b}', 0) for t in targets) / 2048
print(f'M={len(targets)} solutions: P_succès = {p_success:.3f}')

### Analyse de complexité $O(\sqrt{N})$

La complexité quantique $O(\sqrt{N})$ est quadratiquement meilleure que $O(N)$ classique.

In [ ]:
N_vals = 2**np.arange(2, 10)
iters_opt = [int(np.pi / 4 * np.sqrt(N)) for N in N_vals]
classical = N_vals

plt.figure(figsize=(8, 5))
plt.plot(N_vals, classical, label='Classique O(N)', linewidth=2)
plt.plot(N_vals, iters_opt, 'o-', label='Grover O(√N)', linewidth=2)
plt.xlabel("Taille de l'espace N")
plt.ylabel("Nombre d'iterations / appels")
plt.title('Complexité: Grover vs Classique')
plt.legend()
plt.xscale('log', base=2)
plt.yscale('log', base=2)
plt.grid(True)
plt.show()

## Questions

**Q1.** Pour $n=2$ qubits ($N=4$), implémenter Grover avec $M=1$ solution. Combien d'itérations $k$ donnent la probabilité maximale? Que se passe-t-il pour $k > k_{\text{opt}}$?

**Q2.** Tracer $P_{\text{succès}}$ en fonction du nombre d'itérations $k$ pour $n=5$ qubits et $M=1$. Observer le comportement oscillatoire sinusoïdal $P(k) = \sin^2((2k+1)\theta)$ avec $\theta = \arcsin(\sqrt{M/N})$.

In [ ]:
# Q2: Oscillation de la probabilité
n = 5
target = 13
theta = np.arcsin(np.sqrt(1 / 2**n))
k_vals = np.arange(0, 12)
probs = []

for k in k_vals:
    qc = grover_circuit(n, target, k)
    counts = execute(qc, backend, shots=2048).result().get_counts()
    p = counts.get(f'{target:0{n}b}', 0) / 2048
    probs.append(p)

plt.plot(k_vals, probs, 'o-', label='Simulation')
plt.plot(k_vals, np.sin((2 * k_vals + 1) * theta)**2, '--', label='Théorie')
plt.xlabel('Itérations k')
plt.ylabel('P(cible)')
plt.title('Oscillation de Grover')
plt.legend()
plt.grid(True)
plt.show()